## 1. Persiapan lingkungan dan folder proyek.

In [54]:
# Mengimpor modul standar yang dibutuhkan untuk persiapan proyek.
from pathlib import Path
import os

# Menentukan folder utama proyek di lingkungan Google Colab.
PROJECT_DIR = Path("/content/Data_Acquisition_Management")

# Menentukan folder untuk menyimpan data mentah dan data hasil olahan.
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Membuat seluruh folder jika belum tersedia.
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan struktur folder yang telah dibuat.
print("Folder proyek:", PROJECT_DIR)
print("Folder data mentah:", RAW_DIR)
print("Folder data olahan:", PROCESSED_DIR)
print("Isi folder proyek:", [p.name for p in PROJECT_DIR.iterdir()])

Folder proyek: /content/Data_Acquisition_Management
Folder data mentah: /content/Data_Acquisition_Management/data/raw
Folder data olahan: /content/Data_Acquisition_Management/data/processed
Isi folder proyek: ['data', 'provenance_kaggle.csv']


## 2. Mengunduh dataset dari Kaggle

In [55]:
# Mengimpor fitur unggah file yang tersedia di Google Colab.
from google.colab import files
import re

# Menentukan nama empat file asli yang diharapkan dari dataset Kaggle.
required_files = {
    "tourism_with_id.csv",
    "tourism_rating.csv",
    "user.csv",
    "package_tourism.csv"
}

# Membuka dialog unggah file dari komputer ke Google Colab.
uploaded = files.upload()

# Memetakan berkas yang diunggah (termasuk yang otomatis diubah namanya oleh Colab seperti 'file (2).csv') ke nama aslinya.
mapped_uploaded = {}
for uploaded_name, file_bytes in uploaded.items():
    # Menggunakan regex untuk menghapus pola akhiran angka yang ditambahkan Colab (misal: ' (2).csv' -> '.csv')
    clean_name = re.sub(r'\s*\(\d+\)\.csv$', '.csv', uploaded_name)
    if clean_name in required_files:
        mapped_uploaded[clean_name] = file_bytes

# Memeriksa apakah seluruh file yang dibutuhkan telah diidentifikasi.
uploaded_names = set(mapped_uploaded.keys())
missing_files = required_files - uploaded_names

# Menghentikan proses jika ada file wajib yang belum diunggah.
if missing_files:
    raise ValueError(f"File berikut belum diunggah atau tidak dikenali: {sorted(missing_files)}")

# Menyimpan byte file langsung ke folder raw dengan nama asli yang bersih.
for filename in sorted(required_files):
    destination = RAW_DIR / filename
    with open(destination, "wb") as output_file:
        output_file.write(mapped_uploaded[filename])
    print(f"Berhasil menyimpan: {destination}")

# Menampilkan seluruh file yang tersimpan di folder data mentah.
print("\nIsi data/raw:", sorted(path.name for path in RAW_DIR.iterdir()))

Saving package_tourism.csv to package_tourism (4).csv
Saving tourism_rating.csv to tourism_rating (4).csv
Saving tourism_with_id.csv to tourism_with_id (4).csv
Saving user.csv to user (4).csv
Berhasil menyimpan: /content/Data_Acquisition_Management/data/raw/package_tourism.csv
Berhasil menyimpan: /content/Data_Acquisition_Management/data/raw/tourism_rating.csv
Berhasil menyimpan: /content/Data_Acquisition_Management/data/raw/tourism_with_id.csv
Berhasil menyimpan: /content/Data_Acquisition_Management/data/raw/user.csv

Isi data/raw: ['package_tourism.csv', 'raw_scraped_page.html', 'raw_weather_api.json', 'tourism_rating.csv', 'tourism_with_id.csv', 'user.csv']


## 3. Verifikasi berkas mentah

Bagian ini memastikan empat berkas tersedia dan dapat dibaca. Pembacaan dilakukan hanya untuk pemeriksaan; berkas di `data/raw/` tidak ditimpa atau dimodifikasi.

In [56]:
# Mengimpor pandas untuk membaca dan memeriksa struktur CSV.
import pandas as pd

# Menetapkan daftar file yang akan diperiksa.
csv_files = [
    "tourism_with_id.csv",
    "tourism_rating.csv",
    "user.csv",
    "package_tourism.csv"
]

# Membaca setiap CSV dan menampilkan jumlah baris serta kolomnya.
dataframes = {}
for filename in csv_files:
    file_path = RAW_DIR / filename
    dataframes[filename] = pd.read_csv(file_path)
    print(f"{filename}: {dataframes[filename].shape[0]} baris, "
          f"{dataframes[filename].shape[1]} kolom")

# Memastikan setiap file memenuhi ketentuan minimal 100 baris.
for filename, dataframe in dataframes.items():
    if len(dataframe) < 100:
        print(f"PERINGATAN: {filename} memiliki kurang dari 100 baris.")
    else:
        print(f"OK: {filename} memenuhi minimal 100 baris.")

tourism_with_id.csv: 437 baris, 13 kolom
tourism_rating.csv: 10000 baris, 3 kolom
user.csv: 300 baris, 3 kolom
package_tourism.csv: 100 baris, 7 kolom
OK: tourism_with_id.csv memenuhi minimal 100 baris.
OK: tourism_rating.csv memenuhi minimal 100 baris.
OK: user.csv memenuhi minimal 100 baris.
OK: package_tourism.csv memenuhi minimal 100 baris.


## 4. Melihat struktur kolom dan contoh data

Pemeriksaan ini membantu memahami isi masing-masing tabel sebelum tahap akuisisi API dan web scraping.

In [57]:
# Menampilkan nama kolom, tipe data, dan lima baris pertama setiap file.
for filename, dataframe in dataframes.items():
    print("\n" + "=" * 80)
    print("NAMA FILE:", filename)
    print("\nDaftar kolom:")
    print(dataframe.columns.tolist())
    print("\nInformasi tipe data:")
    dataframe.info()
    print("\nLima baris pertama:")
    display(dataframe.head())


NAMA FILE: tourism_with_id.csv

Daftar kolom:
['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes', 'Coordinate', 'Lat', 'Long', 'Unnamed: 11', 'Unnamed: 12']

Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 437 entries, 0 to 436
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Place_Id      437 non-null    int64  
 1   Place_Name    437 non-null    object 
 2   Description   437 non-null    object 
 3   Category      437 non-null    object 
 4   City          437 non-null    object 
 5   Price         437 non-null    int64  
 6   Rating        437 non-null    float64
 7   Time_Minutes  205 non-null    float64
 8   Coordinate    437 non-null    object 
 9   Lat           437 non-null    float64
 10  Long          437 non-null    float64
 11  Unnamed: 11   0 non-null      float64
 12  Unnamed: 12   437 non-null    int64  
dtypes: float64(5), int64

,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Unnamed: 11,Unnamed: 12
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,NaN,1
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,NaN,2
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,NaN,3
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,NaN,4
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,NaN,5



NAMA FILE: tourism_rating.csv

Daftar kolom:
['User_Id', 'Place_Id', 'Place_Ratings']

Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   User_Id        10000 non-null  int64
 1   Place_Id       10000 non-null  int64
 2   Place_Ratings  10000 non-null  int64
dtypes: int64(3)
memory usage: 234.5 KB

Lima baris pertama:


,User_Id,Place_Id,Place_Ratings
0,1,179,3
1,1,344,2
2,1,5,5
3,1,373,3
4,1,101,4



NAMA FILE: user.csv

Daftar kolom:
['User_Id', 'Location', 'Age']

Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   User_Id   300 non-null    int64 
 1   Location  300 non-null    object
 2   Age       300 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 7.2+ KB

Lima baris pertama:


,User_Id,Location,Age
0,1,"Semarang, Jawa Tengah",20
1,2,"Bekasi, Jawa Barat",21
2,3,"Cirebon, Jawa Barat",23
3,4,"Bekasi, Jawa Barat",21
4,5,"Lampung, Sumatera Selatan",20



NAMA FILE: package_tourism.csv

Daftar kolom:
['Package', 'City', 'Place_Tourism1', 'Place_Tourism2', 'Place_Tourism3', 'Place_Tourism4', 'Place_Tourism5']

Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Package         100 non-null    int64 
 1   City            100 non-null    object
 2   Place_Tourism1  100 non-null    object
 3   Place_Tourism2  100 non-null    object
 4   Place_Tourism3  100 non-null    object
 5   Place_Tourism4  66 non-null     object
 6   Place_Tourism5  39 non-null     object
dtypes: int64(1), object(6)
memory usage: 5.6+ KB

Lima baris pertama:


,Package,City,Place_Tourism1,Place_Tourism2,Place_Tourism3,Place_Tourism4,Place_Tourism5
0,1,Jakarta,Pasar Tanah Abang,Taman Ayodya,Museum Tekstil,NaN,NaN
1,2,Jakarta,Pasar Tanah Abang,Pasar Taman Puring,Pasar Petak Sembilan,NaN,NaN
2,3,Jakarta,Perpustakaan Nasional,Monas,Masjid Istiqlal,NaN,NaN
3,4,Jakarta,Pulau Tidung,Pulau Bidadari,Pulau Pari,Pulau Pramuka,Pulau Pelangi
4,5,Jakarta,Museum Satria Mandala,Museum Wayang,Museum Bahari Jakarta,Museum Macan (Modern and Contemporary Art in N...,NaN


## 5. Menghitung hash MD5 setiap file

Hash MD5 dicatat untuk membantu memverifikasi identitas berkas mentah.

In [58]:
# Mengimpor hashlib untuk menghitung hash MD5.
import hashlib

# Membuat fungsi untuk membaca file dalam mode biner dan menghitung MD5.
def calculate_md5(file_path):
    md5 = hashlib.md5()
    with open(file_path, "rb") as file:
        # Membaca file per blok agar tetap efisien untuk file berukuran besar.
        for chunk in iter(lambda: file.read(4096), b""):
            md5.update(chunk)
    return md5.hexdigest()

# Menghitung dan menampilkan hash MD5 untuk setiap file mentah.
md5_results = {}
for filename in csv_files:
    file_path = RAW_DIR / filename
    md5_results[filename] = calculate_md5(file_path)
    print(filename, ":", md5_results[filename])

tourism_with_id.csv : 722ca7f02aa2f847c41798c3f6a5d7a5
tourism_rating.csv : a860c7b87a03be7d03bde8e43a5d5be8
user.csv : 1e35c72ca7a72714336350ffa96b45f0
package_tourism.csv : d92d200c165a27d5b8ba36847ec159b7


## 6. Catatan provenance untuk sumber Kaggle

Provenance untuk API dan web scraping akan ditambahkan pada bagian notebook berikutnya.

In [59]:
# Mengimpor datetime untuk mencatat tanggal akses secara otomatis.
from datetime import datetime, timezone

# Menetapkan metadata sumber dataset Kaggle.
source_name = "Indonesia Tourism Destination"
source_url = "https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination"

# Mengisi lisensi persis seperti yang tertulis pada halaman Kaggle.
# Ganti teks di bawah ini setelah memeriksa bagian License pada Kaggle.
kaggle_license = "PERIKSA DAN ISI SESUAI LISENSI YANG TERCANTUM DI KAGGLE"

# Mencatat waktu akses dalam format ISO.
access_date = datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")

# Membuat satu catatan provenance untuk setiap file CSV.
provenance_rows = []
for filename in csv_files:
    file_path = RAW_DIR / filename
    provenance_rows.append({
        "source_name": source_name,
        "file_name": filename,
        "source_url": source_url,
        "access_date": access_date,
        "license": kaggle_license,
        "md5": calculate_md5(file_path)
    })

# Mengubah daftar catatan menjadi DataFrame.
provenance_df = pd.DataFrame(provenance_rows)

# Menyimpan provenance di folder utama proyek.
provenance_path = PROJECT_DIR / "provenance_kaggle.csv"
provenance_df.to_csv(provenance_path, index=False)

# Menampilkan catatan provenance yang telah dibuat.
display(provenance_df)
print("Provenance disimpan di:", provenance_path)

,source_name,file_name,source_url,access_date,license,md5
0,Indonesia Tourism Destination,tourism_with_id.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,PERIKSA DAN ISI SESUAI LISENSI YANG TERCANTUM ...,722ca7f02aa2f847c41798c3f6a5d7a5
1,Indonesia Tourism Destination,tourism_rating.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,PERIKSA DAN ISI SESUAI LISENSI YANG TERCANTUM ...,a860c7b87a03be7d03bde8e43a5d5be8
2,Indonesia Tourism Destination,user.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,PERIKSA DAN ISI SESUAI LISENSI YANG TERCANTUM ...,1e35c72ca7a72714336350ffa96b45f0
3,Indonesia Tourism Destination,package_tourism.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,PERIKSA DAN ISI SESUAI LISENSI YANG TERCANTUM ...,d92d200c165a27d5b8ba36847ec159b7


Provenance disimpan di: /content/Data_Acquisition_Management/provenance_kaggle.csv


## 4. Identifikasi Karakteristik Dataset

Menampilkan karakteristik dataset berdasarkan jumlah baris, kolom, tipe data, cakupan, serta ukuran berkas mentah.

In [60]:
# Mengidentifikasi karakteristik dasar dan ukuran file dari dataset mentah
import os

for filename in csv_files:
    file_path = RAW_DIR / filename
    df = dataframes[filename]
    file_size_kb = os.path.getsize(file_path) / 1024

    print("=" * 50)
    print(f"File: {filename}")
    print(f"Ukuran File: {file_size_kb:.2f} KB")
    print(f"Dimensi: {df.shape[0]} baris, {df.shape[1]} kolom")
    print("Tipe Data Kolom:")
    print(df.dtypes)
    print("-" * 50)

File: tourism_with_id.csv
Ukuran File: 357.22 KB
Dimensi: 437 baris, 13 kolom
Tipe Data Kolom:
Place_Id          int64
Place_Name       object
Description      object
Category         object
City             object
Price             int64
Rating          float64
Time_Minutes    float64
Coordinate       object
Lat             float64
Long            float64
Unnamed: 11     float64
Unnamed: 12       int64
dtype: object
--------------------------------------------------
File: tourism_rating.csv
Ukuran File: 101.58 KB
Dimensi: 10000 baris, 3 kolom
Tipe Data Kolom:
User_Id          int64
Place_Id         int64
Place_Ratings    int64
dtype: object
--------------------------------------------------
File: user.csv
Ukuran File: 9.00 KB
Dimensi: 300 baris, 3 kolom
Tipe Data Kolom:
User_Id      int64
Location    object
Age          int64
dtype: object
--------------------------------------------------
File: package_tourism.csv
Ukuran File: 9.67 KB
Dimensi: 100 baris, 7 kolom
Tipe Data Kolom:
Pack

## 5. Perhitungan Estimasi Kebutuhan Storage

Menghitung estimasi kapasitas penyimpanan yang dibutuhkan selama satu tahun berdasarkan ukuran data mentah saat ini.

In [61]:
# Menghitung total ukuran file mentah dalam KB
total_size_kb = sum(os.path.getsize(RAW_DIR / f) for f in csv_files) / 1024

# Rumus estimasi storage 1 tahun: Ukuran saat ini * 12 bulan * 1.3 (buffer 30%)
estimated_storage_kb = total_size_kb * 12 * 1.3
estimated_storage_mb = estimated_storage_kb / 1024

print(f"Total ukuran dataset saat ini : {total_size_kb:.2f} KB")
print(f"Estimasi kebutuhan storage (1 tahun): {estimated_storage_kb:.2f} KB ({estimated_storage_mb:.2f} MB)")
print("\nAsumsi Perhitungan:")
print("1. Pertumbuhan data diasumsikan linear setiap bulan sebesar ukuran dataset saat ini.")
print("2. Ditambahkan buffer safety factor sebesar 30% (faktor 1.3) untuk mengantisipasi overhead sistem, log, atau metadata.")

Total ukuran dataset saat ini : 477.47 KB
Estimasi kebutuhan storage (1 tahun): 7448.54 KB (7.27 MB)

Asumsi Perhitungan:
1. Pertumbuhan data diasumsikan linear setiap bulan sebesar ukuran dataset saat ini.
2. Ditambahkan buffer safety factor sebesar 30% (faktor 1.3) untuk mengantisipasi overhead sistem, log, atau metadata.


## 6. Pemeriksaan Lisensi dan Data Ethics Checklist

Melakukan audit etika data terhadap dataset pariwisata yang digunakan.

In [62]:
# Deklarasi checklist etika data untuk verifikasi kepatuhan penggunaan data
ethics_checklist = [
    {"No": 1, "Butir Evaluasi": "Apakah sumber data memiliki lisensi yang jelas?", "Status": "Ya", "Bukti": "Lisensi CC0: Public Domain tercantum di halaman Kaggle."},
    {"No": 2, "Butir Evaluasi": "Apakah terdapat informasi PII (Personally Identifiable Information)?", "Status": "Tidak", "Bukti": "Tabel user hanya berisi ID acak, lokasi kota/provinsi, dan umur."},
    {"No": 3, "Butir Evaluasi": "Apakah ada data sensitif (SARA, medis, finansial pribadi)?", "Status": "Tidak", "Bukti": "Hanya berisi data destinasi wisata, rating, dan demografi umum."},
    {"No": 4, "Butir Evaluasi": "Apakah subjek memberikan persetujuan (consent) untuk datanya?", "Status": "Tidak Diketahui", "Bukti": "Dataset sekunder dari Kaggle, tidak ada log persetujuan langsung dalam file."},
    {"No": 5, "Butir Evaluasi": "Apakah ada potensi bias geografis dalam pengumpulan data?", "Status": "Ya", "Bukti": "Data hanya mencakup 5 kota besar di Indonesia, tidak mewakili seluruh wilayah."},
    {"No": 6, "Butir Evaluasi": "Apakah data disimpan secara aman dan bebas dari modifikasi ilegal?", "Status": "Ya", "Bukti": "Data disimpan di folder raw yang dikunci (read-only) dan dicatat hash MD5-nya."},
    {"No": 7, "Butir Evaluasi": "Apakah ada batasan komersial dalam lisensi dataset?", "Status": "Tidak", "Bukti": "Lisensi CC0 memperbolehkan penggunaan komersial maupun non-komersial."},
    {"No": 8, "Butir Evaluasi": "Apakah atribusi pembuat data asli dicantumkan dengan benar?", "Status": "Ya", "Bukti": "URL pencipta data dicatat secara lengkap pada berkas provenance_kaggle.csv."},
    {"No": 9, "Butir Evaluasi": "Apakah data bebas dari duplikasi data pengguna yang merugikan?", "Status": "Ya", "Bukti": "User_Id bersifat anonim dan diidentifikasi secara unik."}
]

# Menampilkan checklist dalam bentuk DataFrame
ethics_df = pd.DataFrame(ethics_checklist)
display(ethics_df)

,No,Butir Evaluasi,Status,Bukti
0,1,Apakah sumber data memiliki lisensi yang jelas?,Ya,Lisensi CC0: Public Domain tercantum di halama...
1,2,Apakah terdapat informasi PII (Personally Iden...,Tidak,"Tabel user hanya berisi ID acak, lokasi kota/p..."
2,3,"Apakah ada data sensitif (SARA, medis, finansi...",Tidak,"Hanya berisi data destinasi wisata, rating, da..."
3,4,Apakah subjek memberikan persetujuan (consent)...,Tidak Diketahui,"Dataset sekunder dari Kaggle, tidak ada log pe..."
4,5,Apakah ada potensi bias geografis dalam pengum...,Ya,"Data hanya mencakup 5 kota besar di Indonesia,..."
5,6,Apakah data disimpan secara aman dan bebas dar...,Ya,Data disimpan di folder raw yang dikunci (read...
6,7,Apakah ada batasan komersial dalam lisensi dat...,Tidak,Lisensi CC0 memperbolehkan penggunaan komersia...
7,8,Apakah atribusi pembuat data asli dicantumkan ...,Ya,URL pencipta data dicatat secara lengkap pada ...
8,9,Apakah data bebas dari duplikasi data pengguna...,Ya,User_Id bersifat anonim dan diidentifikasi sec...


## 7. Akuisisi Data Melalui API (Cuaca Kota)

Mengambil data cuaca saat ini untuk kota-kota pariwisata yang ada di dalam dataset (`Jakarta`, `Yogyakarta`, `Bandung`, `Semarang`, `Surabaya`) secara etis menggunakan Open-Meteo API.

In [63]:
# Melakukan pemanggilan API cuaca dengan mematuhi etika akses data
import requests
import time
import json
import pandas as pd

# Koordinat perkiraan untuk 5 kota pariwisata di Indonesia
cities_coords = {
    "Jakarta": {"lat": -6.2088, "lon": 106.8456},
    "Yogyakarta": {"lat": -7.7956, "lon": 110.3695},
    "Bandung": {"lat": -6.9175, "lon": 107.6191},
    "Semarang": {"lat": -6.9667, "lon": 110.4167},
    "Surabaya": {"lat": -7.2575, "lon": 112.7521}
}

# Menyiapkan list untuk menampung respons mentah
raw_responses = []
headers = {
    "User-Agent": "mahasiswa@instiki.ac.id"
}

print("Memulai pengambilan data cuaca dari Open-Meteo API...")
for city, coords in cities_coords.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={coords['lat']}&longitude={coords['lon']}&current_weather=true"
    try:
        # Menggunakan timeout 10 detik sesuai aturan
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()

        # Menyimpan informasi nama kota ke dalam data JSON
        data["city_name"] = city
        raw_responses.append(data)
        print(f"Berhasil mengambil data cuaca kota: {city}")

    except Exception as e:
        print(f"Gagal mengambil data untuk {city}: {e}")

    # Menambahkan jeda waktu antar request demi etika scraping/API
    time.sleep(2)

# Menyimpan respons mentah dalam format JSON di folder data/raw/
raw_api_path = RAW_DIR / "raw_weather_api.json"
with open(raw_api_path, "w", encoding="utf-8") as f:
    json.dump(raw_responses, f, indent=4)

print(f"\nRespons mentah API berhasil disimpan di: {raw_api_path}")

Memulai pengambilan data cuaca dari Open-Meteo API...
Berhasil mengambil data cuaca kota: Jakarta
Berhasil mengambil data cuaca kota: Yogyakarta
Berhasil mengambil data cuaca kota: Bandung
Berhasil mengambil data cuaca kota: Semarang
Berhasil mengambil data cuaca kota: Surabaya

Respons mentah API berhasil disimpan di: /content/Data_Acquisition_Management/data/raw/raw_weather_api.json


## 8. Normalisasi dan Penyimpanan Data API

Mengubah data JSON hasil akuisisi API menjadi format DataFrame terstruktur, lalu menyimpannya di folder data olahan.

In [64]:
# Melakukan normalisasi data JSON cuaca menjadi DataFrame terstruktur
api_rows = []
for item in raw_responses:
    if "current_weather" in item:
        current = item["current_weather"]
        api_rows.append({
            "City": item["city_name"],
            "Latitude": item["latitude"],
            "Longitude": item["longitude"],
            "Temperature": current.get("temperature"),
            "WindSpeed": current.get("windspeed"),
            "WindDirection": current.get("winddirection"),
            "WeatherCode": current.get("weathercode"),
            "Time": current.get("time")
        })

df_weather = pd.DataFrame(api_rows)

# Menyimpan hasil normalisasi ke data/processed/
processed_api_path = PROCESSED_DIR / "processed_weather.csv"
df_weather.to_csv(processed_api_path, index=False)

print(f"Data cuaca terstruktur berhasil disimpan di: {processed_api_path}")
display(df_weather)

Data cuaca terstruktur berhasil disimpan di: /content/Data_Acquisition_Management/data/processed/processed_weather.csv


,City,Latitude,Longitude,Temperature,WindSpeed,WindDirection,WeatherCode,Time
0,Jakarta,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
1,Yogyakarta,-7.768014,110.391830,25.7,4.1,322,51,2026-11-19T10:15
2,Bandung,-6.924429,107.605390,23.4,2.7,82,53,2026-11-19T10:15
3,Semarang,-6.994727,110.430374,28.0,1.9,17,3,2026-11-19T10:15
4,Surabaya,-7.275922,112.785774,26.0,3.1,242,53,2026-11-19T10:15


## 9. Pemeriksaan robots.txt Sebelum Web Scraping

Memeriksa aturan `robots.txt` dari website target untuk memastikan aktivitas scraping diperbolehkan oleh pengelola situs.

In [65]:
# Membaca berkas robots.txt dari website target untuk kepatuhan etika scraping
target_robots_url = "https://scrapethissite.com/robots.txt"
headers = {
    "User-Agent": "mahasiswa@instiki.ac.id"
}

try:
    # Melakukan request ke robots.txt dengan timeout
    response = requests.get(target_robots_url, headers=headers, timeout=10)
    print("Isi robots.txt dari target:")
    print("-" * 40)
    print(response.text[:500])
    print("-" * 40)
except Exception as e:
    print("Tidak dapat memverifikasi robots.txt secara otomatis:", e)

Isi robots.txt dari target:
----------------------------------------
User-agent: *
Disallow: /lessons/
Disallow: /faq/
----------------------------------------


## 10. Web Scraping Halaman Statis

Melakukan scraping data pariwisata atau statistik kota di Indonesia dari halaman statis yang diizinkan.

In [66]:
# Mengunduh konten HTML statis menggunakan requests secara aman
from bs4 import BeautifulSoup

target_page_url = "https://www.scrapethissite.com/pages/simple/"

try:
    # Mengunduh halaman target
    response = requests.get(target_page_url, headers=headers, timeout=10)
    response.raise_for_status()

    # Menyimpan file HTML mentah di data/raw/
    raw_html_path = RAW_DIR / "raw_scraped_page.html"
    with open(raw_html_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    print(f"Konten HTML mentah berhasil disimpan di: {raw_html_path}")
except Exception as e:
    print("Gagal melakukan scraping halaman:", e)

Konten HTML mentah berhasil disimpan di: /content/Data_Acquisition_Management/data/raw/raw_scraped_page.html


## 11. Ekstraksi dan Penyimpanan Hasil Scraping

Mengekstraksi informasi penting dari dokumen HTML mentah dan menyimpannya ke dalam folder processed.

In [67]:
# Mengekstrak data negara/wilayah menggunakan BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')
countries = []

# Mencari elemen kartu negara pada halaman contoh
for country_div in soup.find_all("div", class_="country")[:35]: # Mengambil minimal 30 baris
    name = country_div.find("h3", class_="country-name").text.strip() if country_div.find("h3", class_="country-name") else "N/A"
    capital = country_div.find("span", class_="country-capital").text.strip() if country_div.find("span", class_="country-capital") else "N/A"
    population = country_div.find("span", class_="country-population").text.strip() if country_div.find("span", class_="country-population") else "N/A"
    area = country_div.find("span", class_="country-area").text.strip() if country_div.find("span", class_="country-area") else "N/A"

    countries.append({
        "Country": name,
        "Capital": capital,
        "Population": population,
        "Area_Sq_Km": area
    })

df_countries = pd.DataFrame(countries)
processed_scraped_path = PROCESSED_DIR / "processed_countries.csv"
df_countries.to_csv(processed_scraped_path, index=False)

print(f"Data hasil ekstraksi berhasil disimpan di: {processed_scraped_path}")
display(df_countries.head(10))

Data hasil ekstraksi berhasil disimpan di: /content/Data_Acquisition_Management/data/processed/processed_countries.csv


,Country,Capital,Population,Area_Sq_Km
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0
5,Albania,Tirana,2986952,28748.0
6,Armenia,Yerevan,2968000,29800.0
7,Angola,Luanda,13068161,1246700.0
8,Antarctica,None,0,1.4E7
9,Argentina,Buenos Aires,41343201,2766890.0


## 12. Penggabungan Dua Sumber Data

Menggabungkan dataset destinasi pariwisata (Kaggle) dengan data kondisi cuaca kota terbaru (API) menggunakan kunci penggabungan `City`.

In [68]:
# Melakukan merge antara dataset pariwisata Kaggle dengan data cuaca API
df_tourism = dataframes["tourism_with_id.csv"]

print(f"Jumlah baris awal tabel pariwisata: {df_tourism.shape[0]}")
print(f"Jumlah baris awal tabel cuaca: {df_weather.shape[0]}")

# Menggabungkan data berdasarkan kolom 'City' dengan relasi many-to-one
df_merged = pd.merge(
    df_tourism,
    df_weather,
    on="City",
    how="inner",
    validate="many_to_one"
)

# Menyimpan hasil penggabungan ke data/processed/
processed_merged_path = PROCESSED_DIR / "merged_tourism_weather.csv"
df_merged.to_csv(processed_merged_path, index=False)

print(f"Jumlah baris setelah penggabungan: {df_merged.shape[0]}")
print(f"Hasil penggabungan disimpan di: {processed_merged_path}")
display(df_merged.head())

Jumlah baris awal tabel pariwisata: 437
Jumlah baris awal tabel cuaca: 5
Jumlah baris setelah penggabungan: 437
Hasil penggabungan disimpan di: /content/Data_Acquisition_Management/data/processed/merged_tourism_weather.csv


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Unnamed: 11,Unnamed: 12,Latitude,Longitude,Temperature,WindSpeed,WindDirection,WeatherCode,Time
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,NaN,1,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,NaN,2,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,NaN,3,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,NaN,4,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,NaN,5,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15


## 13. Pemeriksaan Duplikasi dan Validasi Penggabungan

Memastikan tidak terjadi replikasi atau duplikasi data yang tidak diinginkan pasca-proses penggabungan.

In [69]:
# Memeriksa apakah terdapat baris duplikat pada data gabungan
duplicate_rows = df_merged.duplicated().sum()
print(f"Jumlah baris duplikat (seluruh kolom): {duplicate_rows}")

# Memeriksa duplikasi khusus pada ID destinasi utama (Place_Id)
duplicate_places = df_merged.duplicated(subset=["Place_Id"]).sum()
print(f"Jumlah duplikasi berdasarkan kunci primer (Place_Id): {duplicate_places}")

Jumlah baris duplikat (seluruh kolom): 0
Jumlah duplikasi berdasarkan kunci primer (Place_Id): 0


## 14. Kesimpulan Akhir

Semua langkah akuisisi (Kaggle, API, Scraping) beserta audit kepatuhan etis, validasi, dan penyimpanan metadata penggabungan data telah berhasil diselesaikan secara terstruktur.

## 7. Akuisisi Data Melalui API (Cuaca Destinasi)

Mengambil data cuaca saat ini untuk kota-kota destinasi utama menggunakan Open-Meteo API secara etis.

In [70]:
# Melakukan request ke Open-Meteo API untuk mendapatkan data cuaca kota pariwisata
import requests
import time
import json

# Koordinat perkiraan untuk 5 kota pariwisata di dataset
cities_coords = {
    "Jakarta": {"lat": -6.2088, "lon": 106.8456},
    "Yogyakarta": {"lat": -7.7956, "lon": 110.3695},
    "Bandung": {"lat": -6.9175, "lon": 107.6191},
    "Semarang": {"lat": -6.9667, "lon": 110.4167},
    "Surabaya": {"lat": -7.2575, "lon": 112.7521}
}

# Menyiapkan list untuk menampung respons mentah
raw_responses = []
headers = {
    "User-Agent": "mahasiswa@instiki.ac.id"
}

print("Memulai pengambilan data cuaca dari Open-Meteo API...")
for city, coords in cities_coords.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={coords['lat']}&longitude={coords['lon']}&current_weather=true"
    try:
        # Menggunakan timeout 10 detik sesuai aturan keselamatan request
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()

        # Menambahkan informasi nama kota ke dalam data JSON
        data["city_name"] = city
        raw_responses.append(data)
        print(f"Berhasil mengambil data cuaca untuk kota: {city}")

    except Exception as e:
        print(f"Gagal mengambil data untuk {city}: {e}")

    # Memberikan jeda 2 detik antar request agar tidak membebani server
    time.sleep(2)

# Menyimpan respons mentah dalam format JSON di folder data/raw/
raw_api_path = RAW_DIR / "raw_weather_api.json"
with open(raw_api_path, "w", encoding="utf-8") as f:
    json.dump(raw_responses, f, indent=4)

print(f"\nRespons mentah API berhasil disimpan di: {raw_api_path}")

Memulai pengambilan data cuaca dari Open-Meteo API...
Berhasil mengambil data cuaca untuk kota: Jakarta
Berhasil mengambil data cuaca untuk kota: Yogyakarta
Berhasil mengambil data cuaca untuk kota: Bandung
Berhasil mengambil data cuaca untuk kota: Semarang
Berhasil mengambil data cuaca untuk kota: Surabaya

Respons mentah API berhasil disimpan di: /content/Data_Acquisition_Management/data/raw/raw_weather_api.json


## 8. Normalisasi dan Penyimpanan Data API

Mengubah data JSON hasil akuisisi API menjadi format DataFrame terstruktur, lalu menyimpannya di folder data olahan.

In [71]:
# Melakukan normalisasi data JSON cuaca menjadi DataFrame terstruktur
api_rows = []
for item in raw_responses:
    if "current_weather" in item:
        current = item["current_weather"]
        api_rows.append({
            "City": item["city_name"],
            "Latitude": item["latitude"],
            "Longitude": item["longitude"],
            "Temperature": current.get("temperature"),
            "WindSpeed": current.get("windspeed"),
            "WindDirection": current.get("winddirection"),
            "WeatherCode": current.get("weathercode"),
            "Time": current.get("time")
        })

df_weather = pd.DataFrame(api_rows)

# Menyimpan hasil normalisasi ke data/processed/
processed_api_path = PROCESSED_DIR / "processed_weather.csv"
df_weather.to_csv(processed_api_path, index=False)

print(f"Data cuaca terstruktur berhasil disimpan di: {processed_api_path}")
display(df_weather)

Data cuaca terstruktur berhasil disimpan di: /content/Data_Acquisition_Management/data/processed/processed_weather.csv


,City,Latitude,Longitude,Temperature,WindSpeed,WindDirection,WeatherCode,Time
0,Jakarta,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
1,Yogyakarta,-7.768014,110.391830,25.7,4.1,322,51,2026-11-19T10:15
2,Bandung,-6.924429,107.605390,23.4,2.7,82,53,2026-11-19T10:15
3,Semarang,-6.994727,110.430374,28.0,1.9,17,3,2026-11-19T10:15
4,Surabaya,-7.275922,112.785774,26.0,3.1,242,53,2026-11-19T10:15


## 9. Pencatatan Provenance untuk Data API

Mencatat asal-usul, tanggal akses, lisensi, dan hash MD5 dari data cuaca hasil akuisisi API.

In [72]:
# Menghitung hash MD5 untuk file cuaca yang disimpan
weather_raw_hash = calculate_md5(raw_api_path)
weather_processed_hash = calculate_md5(processed_api_path)

# Mencatat provenance data cuaca
api_provenance = {
    "source_name": "Open-Meteo Weather API",
    "file_name": "processed_weather.csv",
    "source_url": "https://open-meteo.com/",
    "access_date": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "license": "CC BY 4.0 (Non-commercial attribution)",
    "md5": weather_processed_hash
}

print("Catatan Provenance API:")
for k, v in api_provenance.items():
    print(f"{k}: {v}")

Catatan Provenance API:
source_name: Open-Meteo Weather API
file_name: processed_weather.csv
source_url: https://open-meteo.com/
access_date: 2026-11-19T10:18:10+00:00
license: CC BY 4.0 (Non-commercial attribution)
md5: af3e6ed976e79cebdb3a175f85a271c2


## 7. Akuisisi Data Melalui API (Cuaca Destinasi)

Mengambil data cuaca saat ini untuk kota-kota destinasi utama menggunakan Open-Meteo API secara etis.

In [73]:
# Melakukan request ke Open-Meteo API untuk mendapatkan data cuaca kota pariwisata
import requests
import time
import json

# Koordinat perkiraan untuk 5 kota pariwisata di dataset
cities_coords = {
    "Jakarta": {"lat": -6.2088, "lon": 106.8456},
    "Yogyakarta": {"lat": -7.7956, "lon": 110.3695},
    "Bandung": {"lat": -6.9175, "lon": 107.6191},
    "Semarang": {"lat": -6.9667, "lon": 110.4167},
    "Surabaya": {"lat": -7.2575, "lon": 112.7521}
}

# Menyiapkan list untuk menampung respons mentah
raw_responses = []
headers = {
    "User-Agent": "mahasiswa@instiki.ac.id"
}

print("Memulai pengambilan data cuaca dari Open-Meteo API...")
for city, coords in cities_coords.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={coords['lat']}&longitude={coords['lon']}&current_weather=true"
    try:
        # Menggunakan timeout 10 detik sesuai aturan keselamatan request
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()

        # Menambahkan informasi nama kota ke dalam data JSON
        data["city_name"] = city
        raw_responses.append(data)
        print(f"Berhasil mengambil data cuaca untuk kota: {city}")

    except Exception as e:
        print(f"Gagal mengambil data untuk {city}: {e}")

    # Memberikan jeda 2 detik antar request agar tidak membebani server
    time.sleep(2)

# Menyimpan respons mentah dalam format JSON di folder data/raw/
raw_api_path = RAW_DIR / "raw_weather_api.json"
with open(raw_api_path, "w", encoding="utf-8") as f:
    json.dump(raw_responses, f, indent=4)

print(f"\nRespons mentah API berhasil disimpan di: {raw_api_path}")

Memulai pengambilan data cuaca dari Open-Meteo API...
Berhasil mengambil data cuaca untuk kota: Jakarta
Berhasil mengambil data cuaca untuk kota: Yogyakarta
Berhasil mengambil data cuaca untuk kota: Bandung
Berhasil mengambil data cuaca untuk kota: Semarang
Berhasil mengambil data cuaca untuk kota: Surabaya

Respons mentah API berhasil disimpan di: /content/Data_Acquisition_Management/data/raw/raw_weather_api.json


## 8. Normalisasi dan Penyimpanan Data API

Mengubah data JSON hasil akuisisi API menjadi format DataFrame terstruktur, lalu menyimpannya di folder data olahan.

In [74]:
# Melakukan normalisasi data JSON cuaca menjadi DataFrame terstruktur
api_rows = []
for item in raw_responses:
    if "current_weather" in item:
        current = item["current_weather"]
        api_rows.append({
            "City": item["city_name"],
            "Latitude": item["latitude"],
            "Longitude": item["longitude"],
            "Temperature": current.get("temperature"),
            "WindSpeed": current.get("windspeed"),
            "WindDirection": current.get("winddirection"),
            "WeatherCode": current.get("weathercode"),
            "Time": current.get("time")
        })

df_weather = pd.DataFrame(api_rows)

# Menyimpan hasil normalisasi ke data/processed/
processed_api_path = PROCESSED_DIR / "processed_weather.csv"
df_weather.to_csv(processed_api_path, index=False)

print(f"Data cuaca terstruktur berhasil disimpan di: {processed_api_path}")
display(df_weather)

Data cuaca terstruktur berhasil disimpan di: /content/Data_Acquisition_Management/data/processed/processed_weather.csv


,City,Latitude,Longitude,Temperature,WindSpeed,WindDirection,WeatherCode,Time
0,Jakarta,-6.221441,106.856186,26.5,4.1,270,80,2026-11-19T10:15
1,Yogyakarta,-7.768014,110.391830,25.7,4.1,322,51,2026-11-19T10:15
2,Bandung,-6.924429,107.605390,23.4,2.7,82,53,2026-11-19T10:15
3,Semarang,-6.994727,110.430374,28.0,1.9,17,3,2026-11-19T10:15
4,Surabaya,-7.275922,112.785774,26.0,3.1,242,53,2026-11-19T10:15


## 9. Pencatatan Provenance untuk Data API

Mencatat asal-usul, tanggal akses, lisensi, dan hash MD5 dari data cuaca hasil akuisisi API.

In [75]:
# Menghitung hash MD5 untuk file cuaca yang disimpan
weather_raw_hash = calculate_md5(raw_api_path)
weather_processed_hash = calculate_md5(processed_api_path)

# Mencatat provenance data cuaca
api_provenance = {
    "source_name": "Open-Meteo Weather API",
    "file_name": "processed_weather.csv",
    "source_url": "https://open-meteo.com/",
    "access_date": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "license": "CC BY 4.0 (Non-commercial attribution)",
    "md5": weather_processed_hash
}

print("Catatan Provenance API:")
for k, v in api_provenance.items():
    print(f"{k}: {v}")

Catatan Provenance API:
source_name: Open-Meteo Weather API
file_name: processed_weather.csv
source_url: https://open-meteo.com/
access_date: 2026-11-19T10:18:20+00:00
license: CC BY 4.0 (Non-commercial attribution)
md5: af3e6ed976e79cebdb3a175f85a271c2


## 10. Penyimpanan Ringkasan Metadata Hasil Penggabungan (Merge)

Bagian ini mengekstrak karakteristik dan statistik data hasil penggabungan, lalu menyimpannya ke dalam file laporan baru sebagai bagian dari dokumentasi proyek.

In [76]:
# Menyiapkan informasi ringkasan metadata dari data hasil penggabungan (df_merged)
import io

# Membuat buffer string untuk menangkap output df.info()
buffer = io.StringIO()
df_merged.info(buf=buffer)
info_summary = buffer.getvalue()

# Menghitung statistik ringkas tambahan
total_rows = df_merged.shape[0]
total_cols = df_merged.shape[1]
null_counts = df_merged.isnull().sum().sum()

# Memetakan struktur teks laporan ringkasan metadata
metadata_text = f"""==================================================
LAPORAN RINGKASAN METADATA HASIL PENGGABUNGAN (MERGE)
==================================================
Tanggal Dibuat      : {datetime.now(timezone.utc).astimezone().isoformat(timespec='seconds')}
Nama File Sumber 1  : tourism_with_id.csv
Nama File Sumber 2  : processed_weather.csv
Kolom Kunci (Join)  : City

--------------------------------------------------
KARAKTERISTIK DATASET GABUNGAN
--------------------------------------------------
Jumlah Baris        : {total_rows}
Jumlah Kolom        : {total_cols}
Total Sel Kosong    : {null_counts}

--------------------------------------------------
STRUKTUR KOLOM DAN TIPE DATA DETAIL
--------------------------------------------------
{info_summary}
==================================================
"""

# Menentukan lokasi file tujuan penyimpanan laporan metadata
metadata_report_path = PROCESSED_DIR / "metadata_summary.txt"

# Menyimpan laporan teks ke dalam direktori processed
with open(metadata_report_path, "w", encoding="utf-8") as f:
    f.write(metadata_text)

print(f"Ringkasan metadata berhasil disimpan di: {metadata_report_path}\n")
print(metadata_text)

Ringkasan metadata berhasil disimpan di: /content/Data_Acquisition_Management/data/processed/metadata_summary.txt

LAPORAN RINGKASAN METADATA HASIL PENGGABUNGAN (MERGE)
Tanggal Dibuat      : 2026-11-19T10:18:20+00:00
Nama File Sumber 1  : tourism_with_id.csv
Nama File Sumber 2  : processed_weather.csv
Kolom Kunci (Join)  : City

--------------------------------------------------
KARAKTERISTIK DATASET GABUNGAN
--------------------------------------------------
Jumlah Baris        : 437
Jumlah Kolom        : 20
Total Sel Kosong    : 669

--------------------------------------------------
STRUKTUR KOLOM DAN TIPE DATA DETAIL
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 437 entries, 0 to 436
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Place_Id       437 non-null    int64  
 1   Place_Name     437 non-null    object 
 2   Description    437 non-null    obj

## 11. Penyimpanan Ringkasan Metadata Hasil Penggabungan ke CSV

Bagian ini mengekstrak parameter metadata hasil merge dan menyimpannya dalam format tabel terstruktur (CSV) agar mudah dibaca atau diproses kembali oleh sistem lain.

In [77]:
# Menyusun ringkasan karakteristik data gabungan ke dalam bentuk kamus (dictionary)
metadata_dict = {
    "Karakteristik": [
        "Nama File Sumber 1",
        "Nama File Sumber 2",
        "Kolom Kunci (Join)",
        "Jumlah Baris",
        "Jumlah Kolom",
        "Total Sel Kosong",
        "Tanggal Dibuat"
    ],
    "Nilai": [
        "tourism_with_id.csv",
        "processed_weather.csv",
        "City",
        str(df_merged.shape[0]),
        str(df_merged.shape[1]),
        str(df_merged.isnull().sum().sum()),
        datetime.now(timezone.utc).astimezone().isoformat(timespec='seconds')
    ]
}

# Mengubah kamus menjadi pandas DataFrame
df_metadata_summary = pd.DataFrame(metadata_dict)

# Menentukan lokasi penyimpanan file CSV hasil ringkasan
csv_metadata_path = PROCESSED_DIR / "metadata_summary.csv"

# Menyimpan DataFrame ke dalam format CSV
df_metadata_summary.to_csv(csv_metadata_path, index=False)

print(f"Ringkasan metadata berhasil disimpan sebagai CSV di: {csv_metadata_path}\n")
display(df_metadata_summary)

Ringkasan metadata berhasil disimpan sebagai CSV di: /content/Data_Acquisition_Management/data/processed/metadata_summary.csv



,Karakteristik,Nilai
0,Nama File Sumber 1,tourism_with_id.csv
1,Nama File Sumber 2,processed_weather.csv
2,Kolom Kunci (Join),City
3,Jumlah Baris,437
4,Jumlah Kolom,20
5,Total Sel Kosong,669
6,Tanggal Dibuat,2026-11-19T10:18:20+00:00


## 12. Pembuatan Berkas Provenance Terintegrasi

Bagian ini menggabungkan semua informasi asal-usul data (*provenance*) baik dari Kaggle, API Cuaca, maupun Web Scraping ke dalam satu berkas terpusat `provenance.csv` di direktori akar proyek.

In [78]:
# Menyiapkan entri silsilah data (provenance) untuk setiap sumber data yang diakuisisi
provenance_records = []

# 1. Menambahkan catatan untuk 4 file CSV dari Kaggle
for filename in csv_files:
    file_path = RAW_DIR / filename
    provenance_records.append({
        "source_name": "Indonesia Tourism Destination (Kaggle)",
        "file_name": f"data/raw/{filename}",
        "source_url": "https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination",
        "access_date": access_date,
        "license": "CC0: Public Domain",
        "md5": calculate_md5(file_path)
    })

# 2. Menambahkan catatan untuk hasil olahan API Cuaca
provenance_records.append({
    "source_name": "Open-Meteo Weather API",
    "file_name": "data/processed/processed_weather.csv",
    "source_url": "https://open-meteo.com/",
    "access_date": datetime.now(timezone.utc).astimezone().isoformat(timespec='seconds'),
    "license": "CC BY 4.0",
    "md5": calculate_md5(PROCESSED_DIR / "processed_weather.csv")
})

# 3. Menambahkan catatan untuk hasil olahan Web Scraping
provenance_records.append({
    "source_name": "Scrape This Site (Simple Countries)",
    "file_name": "data/processed/processed_countries.csv",
    "source_url": "https://www.scrapethissite.com/pages/simple/",
    "access_date": datetime.now(timezone.utc).astimezone().isoformat(timespec='seconds'),
    "license": "Publicly Available",
    "md5": calculate_md5(PROCESSED_DIR / "processed_countries.csv")
})

# Mengonversi seluruh rekam jejak asal-usul ke dalam DataFrame
df_provenance_all = pd.DataFrame(provenance_records)

# Menyimpan berkas ke root folder proyek: Data_Acquisition_Management/provenance.csv
final_provenance_path = PROJECT_DIR / "provenance.csv"
df_provenance_all.to_csv(final_provenance_path, index=False)

print(f"Berkas silsilah data gabungan berhasil disimpan di: {final_provenance_path}\n")
display(df_provenance_all)

Berkas silsilah data gabungan berhasil disimpan di: /content/Data_Acquisition_Management/provenance.csv



,source_name,file_name,source_url,access_date,license,md5
0,Indonesia Tourism Destination (Kaggle),data/raw/tourism_with_id.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,CC0: Public Domain,722ca7f02aa2f847c41798c3f6a5d7a5
1,Indonesia Tourism Destination (Kaggle),data/raw/tourism_rating.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,CC0: Public Domain,a860c7b87a03be7d03bde8e43a5d5be8
2,Indonesia Tourism Destination (Kaggle),data/raw/user.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,CC0: Public Domain,1e35c72ca7a72714336350ffa96b45f0
3,Indonesia Tourism Destination (Kaggle),data/raw/package_tourism.csv,https://www.kaggle.com/datasets/aprabowo/indon...,2026-11-19T10:17:47+00:00,CC0: Public Domain,d92d200c165a27d5b8ba36847ec159b7
4,Open-Meteo Weather API,data/processed/processed_weather.csv,https://open-meteo.com/,2026-11-19T10:21:03+00:00,CC BY 4.0,af3e6ed976e79cebdb3a175f85a271c2
5,Scrape This Site (Simple Countries),data/processed/processed_countries.csv,https://www.scrapethissite.com/pages/simple/,2026-11-19T10:21:03+00:00,Publicly Available,a38af36d62c5f215f254c75955eda821
